<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/Part2_Experiments_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 1 — Gradual Unfreezing on ResNet50

Will be setting up our data loaders for our 3 experiments

1.   Linear Probing
2.   Full Fine-Tuning
3.   Progressive Unfreezing

But the three expriments share the same "beginning"



## 1. Setup and imports

Mounts Drive, imports libraries, fixes random seeds for reproducibility, and confirm GPU usage

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.multiprocessing as mp
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'gpu: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
device: cuda
gpu: NVIDIA A100-SXM4-80GB


## 2. Paths and shared training config


In [2]:
# central config — anything we might tweak lives here
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
INDEX_DIR = DRIVE_ROOT / 'sample_index'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'exp1_gradual_unfreezing'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# training config shared across A, B, C
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 0
NUM_EPOCHS = 20
LR = 1e-3              # base lr; backbone lr will be set lower in B and C
WEIGHT_DECAY = 1e-4
PATIENCE = 5           # early stopping on val loss

# imagenet stats — required because the backbone is pretrained on imagenet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'results dir: {RESULTS_DIR}')

results dir: /content/drive/MyDrive/Colab Notebooks/AdvancedDL/results/exp1_gradual_unfreezing


In [ ]:
import shutil
# takes 8 mins to run
LOCAL_ROOT = Path('/content/dataset_local')
DRIVE_RAW = DRIVE_ROOT / 'raw'

EXPECTED = {'train': 4654, 'val': 1125, 'test': 1124}

def copy_split(split):
    src = DRIVE_RAW / split / 'images'
    dst = LOCAL_ROOT / split / 'images'
    dst.mkdir(parents=True, exist_ok=True)
    existing = len(list(dst.glob('*.png')))
    if existing == EXPECTED[split]:
        print(f'  {split}: already complete ({existing} images)')
        return
    print(f'  {split}: copying {EXPECTED[split]} images...')
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    elapsed = time.time() - t0
    n = len(list(dst.glob('*.png')))
    print(f'  {split}: {n} images copied in {elapsed:.0f}s')

print('copying dataset to local SSD...')
for split in ['train', 'val', 'test']:
    copy_split(split)
print('done.')

copying dataset to local SSD...
  train: copying 4654 images...


## 3. Load sample index from EDA

Reads the three JSONs (train/val/test) and the vocab the EDA produced. No re-processing here: the EDA already filtered everything and left the ground truth ready to consume. Also loads the class_to_idx mapping the model will use internally.

In [ ]:
with open(INDEX_DIR / 'samples_train.json') as f:
    samples_train = json.load(f)
with open(INDEX_DIR / 'samples_val.json') as f:
    samples_val = json.load(f)
with open(INDEX_DIR / 'samples_test.json') as f:
    samples_test = json.load(f)
with open(INDEX_DIR / 'class_vocab.json') as f:
    vocab = json.load(f)

CLASS_TO_IDX = vocab['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in vocab['idx_to_class'].items()}
NUM_CLASSES = len(CLASS_TO_IDX)
CLASSES = vocab['keep_classes']

# rewrite paths from drive to local ssd
DRIVE_PREFIX = str(DRIVE_RAW)
LOCAL_PREFIX = str(LOCAL_ROOT)
for split_samples in [samples_train, samples_val, samples_test]:
    for s in split_samples:
        s['img_path'] = s['img_path'].replace(DRIVE_PREFIX, LOCAL_PREFIX)

# sanity check  first sample's path should now point to /content/
print(f'first train sample path: {samples_train[0]["img_path"]}')
print(f'classes: {NUM_CLASSES}  ({CLASSES})')
print(f'train: {len(samples_train):,}  val: {len(samples_val):,}  test: {len(samples_test):,}')

## 4. Class weights for imbalance
Classes are imbalanced ~35x (rack vs sign). The EDA decided to handle this with weights in the loss, not undersampling. The formula total / (num_classes * count[i]) gives weight 1.0 to an average-frequency class, high weight to rare ones (sign ~11.7), low to common ones (rack ~0.34). These weights feed into CrossEntropyLoss.

In [ ]:
# weight[i] = total / (num_classes * count[i])
# a class with average frequency gets weight 1.0; rare classes get higher weight
counts = Counter(s['class_name'] for s in samples_train)
total = sum(counts.values())
weights = torch.zeros(NUM_CLASSES)
for cls, idx in CLASS_TO_IDX.items():
    weights[idx] = total / (NUM_CLASSES * counts[cls])

print('class weights:')
for cls, idx in sorted(CLASS_TO_IDX.items(), key=lambda x: x[1]):
    print(f'  {idx:2d} {cls:12s} count={counts[cls]:6,d}  weight={weights[idx]:.3f}')

class_weights = weights.to(device)

## 5. Dataset class
Each sample in the JSON is one bbox inside a larger image; the Dataset opens the image, crops with the bbox, and applies transforms. Cropping happens on the fly (no pre-generated files on disk), which keeps the original dataset untouched and lets us change transforms without re-processing anything.

In [ ]:
class WarehouseCrops(Dataset):
    """one sample = one bbox crop from a warehouse image."""
    def __init__(self, samples, transform):
        self.samples = samples
        self.tf = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        img = Image.open(s['img_path']).convert('RGB')
        x1, y1, x2, y2 = s['bbox']
        crop = img.crop((x1, y1, x2, y2))
        x = self.tf(crop)
        y = s['label']
        return x, y

## 6. Trasnforms and conforms datasets

Two pipelines: train_tf with light augmentation (horizontal flip + mild color jitter, defensible because we're not investigating augmentation) and a deterministic eval_tf for val/test. Both resize to 224×224 with stretch (EDA decision) and normalize with ImageNet stats.

In [ ]:
# train: stretch resize (no letterbox, decided in EDA) + mild augmentation
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# eval: no augmentation, deterministic
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = WarehouseCrops(samples_train, train_tf)
val_ds = WarehouseCrops(samples_val, eval_tf)
test_ds = WarehouseCrops(samples_test, eval_tf)

print(f'train_ds: {len(train_ds):,}')
print(f'val_ds:   {len(val_ds):,}')
print(f'test_ds:  {len(test_ds):,}')

## 7. DataLoaders and visual sanity check
DataLoaders batch the samples and load them in parallel via workers. pin_memory=True speeds up the CPU→GPU transfer. We pull one batch to verify three things: shapes are as expected, pixel values fall in the normalized range (~[-2, 3]), and crops visually match their labels.

In [ ]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f'train batches: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

## 7.1 — In-memory dataset utilities

To bypass the data loading bottleneck we observed (DataLoader workers either deadlocking or running too slow with `num_workers=0`), we pre-load all crops directly into RAM as `uint8` tensors. After pre-loading, the "DataLoader" is just tensor indexing — no PIL, no file I/O, no transforms during training. Normalization happens on the GPU per batch instead.

The `preload_to_ram` function takes a sample list and an optional `n_max` cap, useful for quick smoke tests before committing to the full dataset.

In [ ]:
# pre-load crops to ram as uint8 tensors
# uint8 keeps memory usage low (~150 KB per crop instead of 600 KB as float32)
def preload_to_ram(samples, n_max=None):
    n = len(samples) if n_max is None else min(n_max, len(samples))
    # allocate the full tensor up front so we don't pay for repeated reallocations
    imgs = torch.empty((n, 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
    labels = torch.empty((n,), dtype=torch.long)

    t0 = time.time()
    for i in range(n):
        s = samples[i]
        img = Image.open(s['img_path']).convert('RGB')
        x1, y1, x2, y2 = s['bbox']
        crop = img.crop((x1, y1, x2, y2))
        crop = crop.resize((IMG_SIZE, IMG_SIZE))
        # convert PIL to tensor in HWC uint8, then permute to CHW
        arr = np.asarray(crop)
        imgs[i] = torch.from_numpy(arr).permute(2, 0, 1)
        labels[i] = s['label']
        # progress print every 1000 samples — gives us a heartbeat we can monitor
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / elapsed
            eta = (n - i - 1) / rate
            print(f'  loaded {i+1:>6,d} / {n:,}  ({rate:.0f}/s, eta {eta:.0f}s)')
    print(f'done: {n:,} samples in {(time.time()-t0):.0f}s')
    return imgs, labels


# in-memory dataset — getitem is just tensor indexing, normalization happens here
# imagenet mean/std as tensors so we can normalize without going through pil
class InMemoryDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs
        self.labels = labels
        # buffer normalization stats so we don't recreate them per call
        self.mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1) * 255
        self.std = torch.tensor(IMAGENET_STD).view(3, 1, 1) * 255

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        # convert uint8 [0,255] to normalized float
        x = self.imgs[i].float()
        x = (x - self.mean) / self.std
        return x, self.labels[i]


print('preload utilities defined')

## 8. Shared helpers — Model, freeze utilities, train/eval loops

Antes de las variantes, definimos lo que las tres comparten: cómo construir el ResNet50, helpers para congelar/descongelar capas, y los loops de train/eval. Cada variante después usa estos helpers con su propio setup.

In [ ]:
import torchvision.models as models
from torch.amp import autocast, GradScaler

# build resnet50 with imagenet pretrained weights
# v2 weights are the most recent and best from torchvision
def build_model(num_classes):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    # replace the final layer (originally 1000 classes for imagenet) with our 11
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# turn off gradients for every parameter — the model becomes a fixed feature extractor
def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

# turn gradients back on for any parameter whose name starts with one of the prefixes
# example: unfreeze(model, ['layer4']) re-enables training for the last resnet block
def unfreeze(model, prefixes):
    for n, p in model.named_parameters():
        if any(n.startswith(pre) for pre in prefixes):
            p.requires_grad = True

# quick sanity check on how many params will actually receive gradient updates
def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

# one full pass over the training set
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb in loader:
        # move batch to gpu — non_blocking helps overlap data transfer with compute
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        # set_to_none is faster than zeroing — frees the gradient memory
        optimizer.zero_grad(set_to_none=True)
        # autocast runs ops in float16 where safe, float32 where precision matters
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        # scaler prevents float16 underflow during backprop
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # track running totals so we can report epoch averages
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n

# one full pass over the validation set, no gradients
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    # per class tracking so we can see whether sign and other rare classes are learning
    per_class_correct = torch.zeros(NUM_CLASSES)
    per_class_total = torch.zeros(NUM_CLASSES)
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        preds = logits.argmax(1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_n += xb.size(0)
        # count correct predictions per class for the per-class accuracy table
        for c in range(NUM_CLASSES):
            mask = (yb == c)
            per_class_total[c] += mask.sum().item()
            per_class_correct[c] += (preds[mask] == c).sum().item()
    # clamp avoids divide-by-zero if a class has no samples in val (shouldn't happen but safe)
    per_class_acc = (per_class_correct / per_class_total.clamp(min=1)).tolist()
    return total_loss / total_n, total_correct / total_n, per_class_acc

print('helpers defined')

## 8.1 Smoke test

Before committing to pre-loading all 225k training samples (~33 GB and 15-20 minutes), we test the full pipeline on a small slice: pre-load 5,000 samples, train 2 epochs of variant A with progress prints every 50 batches. If this completes in a few minutes with visible output, the approach works and we can scale up confidently. If it hangs or fails, we catch it in minutes instead of after a long pre-load.

In [ ]:
# pre-load a small slice to verify the pipeline works
print('pre-loading 5,000 samples (smoke test)...')
smoke_imgs, smoke_labels = preload_to_ram(samples_train, n_max=5000)
print(f'memory used: {smoke_imgs.element_size() * smoke_imgs.nelement() / 1e9:.2f} GB')

smoke_ds = InMemoryDataset(smoke_imgs, smoke_labels)
smoke_loader = DataLoader(smoke_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f'smoke loader: {len(smoke_loader)} batches')

# build a quick model + optimizer just to test the train step
smoke_model = build_model(NUM_CLASSES).to(device)
freeze_all(smoke_model)
for p in smoke_model.fc.parameters():
    p.requires_grad = True
smoke_opt = torch.optim.AdamW(
    [p for p in smoke_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
smoke_crit = nn.CrossEntropyLoss(weight=class_weights)
smoke_scaler = GradScaler()

# 2 epochs with frequent prints so we see real-time progress
print('\nrunning 2 epochs of variant A on smoke set...')
for epoch in range(2):
    smoke_model.train()
    t0 = time.time()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for i, (xb, yb) in enumerate(smoke_loader):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        smoke_opt.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = smoke_model(xb)
            loss = smoke_crit(logits, yb)
        smoke_scaler.scale(loss).backward()
        smoke_scaler.step(smoke_opt)
        smoke_scaler.update()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n += xb.size(0)
        # progress print every 10 batches — this is what tells us if it's alive
        if (i + 1) % 10 == 0:
            print(f'  epoch {epoch} batch {i+1:>3d}/{len(smoke_loader)}  '
                  f'loss {total_loss/total_n:.4f}  acc {total_correct/total_n:.4f}')
    print(f'epoch {epoch} done in {time.time()-t0:.1f}s')

print('\nsmoke test passed.')

pre-loading 5,000 samples (smoke test)...
  loaded  1,000 / 5,000  (75/s, eta 53s)
  loaded  2,000 / 5,000  (74/s, eta 40s)
  loaded  3,000 / 5,000  (74/s, eta 27s)
  loaded  4,000 / 5,000  (74/s, eta 14s)
  loaded  5,000 / 5,000  (74/s, eta 0s)
done: 5,000 samples in 67s
memory used: 0.75 GB
smoke loader: 40 batches

running 2 epochs of variant A on smoke set...
  epoch 0 batch  10/40  loss 2.0298  acc 0.4523
  epoch 0 batch  20/40  loss 1.7273  acc 0.5602
  epoch 0 batch  30/40  loss 1.4933  acc 0.6320
  epoch 0 batch  40/40  loss 1.3598  acc 0.6762
epoch 0 done in 6.7s
  epoch 1 batch  10/40  loss 0.6491  acc 0.8797
  epoch 1 batch  20/40  loss 0.6015  acc 0.8840
  epoch 1 batch  30/40  loss 0.5709  acc 0.8943
  epoch 1 batch  40/40  loss 0.5464  acc 0.8994
epoch 1 done in 6.1s

smoke test passed.
